# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os
import shutil
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('data'):
    shutil.rmtree('data')
    os.makedirs('data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    CatsUserID = 2496, 
    days_back = 365 * 2,
    out_of_scope = ['-80', 'Cryo tank', 'Water'],
    coris_enabled = True,
    hobolink_enabled = True,
    conserv_enabled = False, # out of scope for this project (just adding Hobolink). 
    testing = True
)

DEBUG: Enabled data sources: ['Coris', 'Hobolink']


Gathering Hobolink readings: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:05<00:00,  1.49s/it]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-11-05 11:12:15,577 - EnvironmentData - INFO - Created Hobolink client
2025-11-05 11:12:15,577 - EnvironmentData - INFO - Enabled data sources: ['Coris', 'Hobolink']
2025-11-05 11:12:15,577 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/cats/user/?ApiKey=XXXX&CatsUserID=XXXX
2025-11-05 11:12:17,630 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21373&ReadingType=SensorReadingF&StartUTC=1699294335&EndUTC=1762366335&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-05 11:12:24,860 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21375&ReadingType=SensorReadingF&StartUTC=1699294335&EndUTC=1762366335&MinReadingSpacing=600&RequestedOutputFormat=raw
2025-11-05 11:12:31,250 - EnvironmentData - INFO - API call: https://cats.corismonitoring.com/api/sensor/historical/?ApiKey=XXXX&SensorID=21377&ReadingType=SensorReadingF&StartUTC

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1699294335,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.639999,null
1699294935,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.620003,null
1699295535,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.580002,null
1699296135,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.580002,null
1699296735,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.639999,null


In [4]:
polars.read_parquet('data/sensor_readings.parquet').filter(polars.col("Source") == "Hobolink").head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh
i64,str,str,str,str,str,str,f32,f32
1736967600,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-2""",null,"""RH""",null,10.899519
1736969400,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-2""",null,"""RH""",null,10.269321
1736345700,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-2""",null,"""RH""",null,15.454337
1736374500,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-2""",null,"""RH""",null,16.137941
1736377200,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179174-2""",null,"""RH""",null,15.066758


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [5]:
envdt.get_current_readings()

# Data is read into new-readings folder for consolidation at the end of the day.
import os
filename = os.listdir('data/new-readings')[0]
print(filename)
polars.read_parquet('data/new-readings/' + filename).sample(5)

C:\Users\Bryce\Documents\arbaiza\environmental-sensor-poc\EnvironmentData.py:220: UserWarning: get_current_readings_coris validation errors : SensorReadingUTC is 5 minutes old

  warnings.warn(msg)


1762366389.parquet


SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC
i64,str,str,str,str,str,str,f32,f32,i32
1762366392,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179174-2""","""RX Station 1_RH""","""RH""",null,39.790951,1762366391
1762366242,"""Coris""","""coris:12169""","""Peabody TH-L Diorama Room 304 …","""coris:21376""","""PYPM__0300302SET____ RH YPM 30…","""Humidity""",null,47.790001,1762366389
1762366392,"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-1""","""RX Station 2_Temperature""","""Temperature""",70.103424,null,1762366391
1762366242,"""Coris""","""coris:12169""","""Peabody TH-L Diorama Room 304 …","""coris:21375""","""PYPM__0300302SET____ Temp YPM …","""Temperature""",68.099998,null,1762366389
1762366392,"""Hobolink""","""hobolink:10740550""","""ICSC__010C149_______""","""hobolink:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""",null,51.803699,1762366391


# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [6]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
print(os.listdir('data/new-readings'))

C:\Users\Bryce\Documents\arbaiza\environmental-sensor-poc\EnvironmentData.py:220: UserWarning: consolidate_readings validation errors : Count of sensors missing from historical data: 38.

  warnings.warn(msg)


[]


**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [7]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [8]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC,SensorReadingUTC_SecondsFromPrior
i64,str,str,str,str,str,str,f32,f32,i32,i64
1699294335,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.639999,null,null,null
1699294935,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.620003,null,null,600
1699295535,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.580002,null,null,600
1699296135,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.580002,null,null,600
1699296735,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",69.639999,null,null,600


In [9]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,QueryUTC,SensorReadingUTC_SecondsFromPrior
i64,str,str,str,str,str,str,f32,f32,i32,i64
1762360200,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-1""",null,"""Temperature""",71.532005,null,null,900
1762361100,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-1""",null,"""Temperature""",71.570618,null,null,900
1762362000,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-1""",null,"""Temperature""",71.60923,null,null,900
1762366392,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.60923,null,1762366391,4392
1762366392,"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-2""","""RX Station 1_RH""","""RH""",null,38.583961,1762366391,null


In [10]:
# Device Readings.
device_readings = polars.read_parquet('data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,f32,f32
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1699294335,null,null,50.049999
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1699294935,null,null,50.200001
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1699295535,null,null,50.240002
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1699296135,null,null,50.240002
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""PYPM__0100104SET____ RH YPM 10…","""Humidity, Temperature""",1699296735,null,null,49.830002


In [11]:
# Sensors
sensors = polars.read_parquet('data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str
"""Hobolink""","""hobolink:10740550-10740550-2""","""ICSC__010C149________RH""","""RH""","""hobolink:10740550""","""RH""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Hobolink""","""hobolink:10740550-10740550-1""","""ICSC__010C149________Temperatu…","""Temperature""","""hobolink:10740550""","""Temperature""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Coris""","""coris:21378""","""RH KGL 21_D0B2""","""Humidity""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Coris""","""coris:21377""","""Temp KGL 21_D0B2""","""Temperature""","""coris:12167""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Hobolink""","""hobolink:22202142-22179175-2""","""RX Station 1_RH""","""RH""","""hobolink:22202142""","""RH""","""Station""","""Unknown""","""1""","""Not Indicated"""


In [12]:
# Devices. 
devices = polars.read_parquet('data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""Hobolink""","""hobolink:10740550""","""ICSC__010C149_______""","""hobolink:10740550-10740550-2, …","""ICSC__010C149________RH, ICSC_…","""RH, Temperature""","""Temperature""","""FLOATER""","""Unknown""","""FLOATER""",null
"""Coris""","""coris:12167""","""Peabody TH-L Upper Great Hall …","""coris:21378, coris:21377""","""RH KGL 21_D0B2, Temp KGL 21_D0…","""Humidity, Temperature""","""D0B2""","""KGL""","""Kline Geology Laboratory""","""21""","""Not Indicated"""
"""Hobolink""","""hobolink:22202142""","""RX Station 1""","""hobolink:22202142-22179175-2, …","""RX Station 1_RH, RX Station 1_…","""RH, RH, Temperature, Temperatu…","""Temperature""","""Station""","""Unknown""","""1""","""Not Indicated"""
"""Hobolink""","""hobolink:22202141""","""RX Station 2""","""hobolink:22202141-22179175-2, …","""RX Station 2_RH, RX Station 2_…","""RH, Temperature""","""RH""","""Station""","""Unknown""","""2""","""Not Indicated"""
"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21374, coris:21373""","""RH YPM 104_D444 , Temp YPM 104…","""Humidity, Temperature""","""D444""","""YPM""","""Yale Peabody Museum""","""104""","""Not Indicated"""


In [13]:
# UTC Date/Time Info
utcs = polars.read_parquet('data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1753743360,2025-07-28 16:56:00,2025-07-28 12:56:00 EDT,2025-07-28,12:56:00,2025,7,"""Monday""",1,12,0,"""PM"""
1753481220,2025-07-25 16:07:00,2025-07-25 12:07:00 EDT,2025-07-25,12:07:00,2025,7,"""Friday""",5,12,0,"""PM"""
1707081735,2024-02-04 14:22:15,2024-02-04 09:22:15 EST,2024-02-04,09:22:15,2024,2,"""Sunday""",7,9,9,"""AM"""
1726742535,2024-09-19 04:42:15,2024-09-19 00:42:15 EDT,2024-09-19,00:42:15,2024,9,"""Thursday""",4,0,0,"""AM"""
1746403335,2025-05-04 18:02:15,2025-05-04 14:02:15 EDT,2025-05-04,14:02:15,2025,5,"""Sunday""",7,14,2,"""PM"""


In [14]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Coris""",2023-11-06,"""coris:21373""",107,7424.549805,0.0,69.129997,null,69.639999,null
"""Coris""",2023-11-06,"""coris:21374""",107,0.0,5498.779785,null,49.330002,null,54.720001
"""Coris""",2023-11-06,"""coris:21375""",107,7305.25,0.0,67.93,null,68.589996,null
"""Coris""",2023-11-06,"""coris:21376""",107,0.0,5615.720215,null,48.639999,null,58.990002
"""Coris""",2023-11-06,"""coris:21377""",107,8009.709961,0.0,73.089996,null,76.330002,null


In [15]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Coris""",2023-11-06,"""coris:12162""",214,7424.549805,5498.779785,69.129997,49.330002,69.639999,54.720001
"""Coris""",2023-11-06,"""coris:12167""",214,8009.709961,4478.399902,73.089996,40.07,76.330002,42.619999
"""Coris""",2023-11-06,"""coris:12169""",214,7305.25,5615.720215,67.93,48.639999,68.589996,58.990002
"""Coris""",2023-11-07,"""coris:12162""",288,9938.169922,7804.0,68.519997,51.869999,69.349998,57.34
"""Coris""",2023-11-07,"""coris:12167""",288,10705.299805,6328.009766,73.260002,42.200001,75.379997,45.27


In [16]:
# Differentiate historical vs. cron readings by filtering on QueryUTC = NULL.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('data/device_readings.parquet') 
    WHERE QueryUTC is null
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,SensorReadingF,SensorReadingRh
0,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1699294335,<NA>,NaN,50.049999
1,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1699294935,<NA>,NaN,50.200001
2,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1699295535,<NA>,NaN,50.240002
3,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1699296135,<NA>,NaN,50.240002
4,Coris,coris:12162,Peabody TH-L Mammal Hall D444,"coris:21374, coris:21373","PYPM__0100104SET____ RH YPM 104_D444 , PYPM__0...","Humidity, Temperature",1699296735,<NA>,NaN,49.830002


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.